# Template Energy Experiment — hand-painted geometric priors

**Frozen U-Net → anchored projected-gradient refinement → learned strengths of 8 fixed painted 21×21 feature templates.**

The two old separator/boundary filter families and learned kernel parameters are **not used**. The original three-phase Modica–Mortola term remains unchanged. Each PNG is rotated into 16 orientations, the responses are combined via maximum → ReLU → negative squared reward, and each template has one bounded trainable strength shared across every refinement iteration.

The input modes, stochastic blurred/spectral corruptions, detached warm-up, 4-step differentiable rollout, SGD, weighted CE + Dice, and 8-connected instance F1/merge/split validation follow Experiment B. This notebook requires a **new checkpoint output directory**; previous Experiment B checkpoints are incompatible.

## 1. Upload ZIP and mount Drive

In [ ]:
%pip -q install psfmodels scipy scikit-image numba matplotlib pandas pillow

In [ ]:
from google.colab import drive, files
from pathlib import Path
import sys, zipfile

drive.mount('/content/drive')
print('Upload template_energy_experiment_bundle.zip')
uploaded = files.upload()
zip_name = next((k for k in uploaded if k.endswith('.zip')), None)
if zip_name is None:
    raise ValueError('Upload the complete template_energy_experiment_bundle.zip')
with zipfile.ZipFile(zip_name) as zf:
    zf.extractall('/content')
PROJECT = Path('/content/template_energy_experiment')
assert (PROJECT / 'template_energy.py').exists()
assert len(list((PROJECT / 'templates').glob('*.png'))) == 8
sys.path.insert(0, str(PROJECT))
print('Template directory:', PROJECT / 'templates')

## 2. Verify the painted templates and initial strengths

In [ ]:
from template_energy import TemplateEnergy
from mm_fixed_compatible import Config
import matplotlib.pyplot as plt
from PIL import Image
import torch

paths = sorted((PROJECT / 'templates').glob('*.png'))
fig, axes = plt.subplots(2, 4, figsize=(12, 6), constrained_layout=True)
for ax, path in zip(axes.flat, paths):
    ax.imshow(Image.open(path), interpolation='nearest')
    ax.set_title(path.stem)
    ax.axis('off')
plt.show()

base_cfg = Config(lam=.2, epsilon=.5, well_cubic_weight=1., barrier_weight=0., boundary_weight=0.)
energy = TemplateEnergy(base_cfg, template_dir=PROJECT / 'templates', orientations=16,
                        strength_cap=.3, initial_strength=.05)
print('Trainable parameters:', sum(p.numel() for p in energy.parameters() if p.requires_grad))
print('Initial template strengths:', energy.diagnostics())

## 3. Training configuration

In [ ]:
from train_template_energy import TrainBConfig, train

BASELINE_PATH = Path('/content/drive/MyDrive/Colab Notebooks/proj_26/checkpoints_unet_baseline_v1_boosted_ce/latest.pt')
assert BASELINE_PATH.exists(), f'Could not find {BASELINE_PATH}'

cfg = TrainBConfig(
    checkpoint=str(BASELINE_PATH),
    output_dir='/content/drive/MyDrive/Colab Notebooks/proj_26/template_energy_results',
    device='cuda' if torch.cuda.is_available() else 'cpu',
    seed=2400,
    train_seed=200000,
    val_seed=300000,
    train_volumes=8,
    val_volumes=3,
    N=192, z_slices=16, z_ratio=2, n_seeds=45,
    class_probs=(2/5,1/5,2/5),
    train_domain='a', val_domain='a',
    template_dir=str(PROJECT / 'templates'),
    orientations=16,
    strength_cap=0.3,
    initial_strength=0.05,
    anchor_mu=0.4,
    step_size=0.02,
    updates=250,
    learning_rate=1.0,  # SGD; change if desired; start conservatively
    momentum=0.0,
    weight_decay=1e-5,
    batch_size=2,
    warmups=(0,20,80,160,300),
    differentiable_steps=4,
    source_mode='mixed',
    prob_unet_in_mixed=0.5,
    gt_blur_range=(0.0,1.5),
    gt_spectral_range=(0.4,1.2),
    restart_probability=0.5,
    restart_blur_range=(0.0,0.8),
    restart_spectral_range=(0.0,0.5),
    strength_drift_penalty=0.0,
    extra_strength_penalty=1e-3,
    grad_clip=5.0,
    eval_every=25,
    val_horizons=(0,50,150,300,400),
    validation_max_slices=8,
)
print(cfg)

## 4. Optional small pilot

In [ ]:
from dataclasses import replace
# Uncomment to test a short run; it saves into a separate subfolder.
# pilot = replace(cfg, output_dir=str(Path(cfg.output_dir)/'pilot'),
#                 train_volumes=1, val_volumes=1, updates=2,
#                 warmups=(0,2), val_horizons=(0,2), validation_max_slices=1)
# pilot_model, pilot_history, pilot_validation = train(pilot)

## 5. Train and evaluate

In [ ]:
model, history, val_history = train(cfg)
print('Final strengths:')
for name, strength in model.diagnostics().items():
    print(f'{name:24s}: {strength:.6f}')

## 6. Curves and strength changes

In [ ]:
import pandas as pd
from pathlib import Path
out = Path(cfg.output_dir)
train_df = pd.read_csv(out/'train_history.csv')
val_df = pd.read_csv(out/'validation_history.csv')
fig, axes = plt.subplots(1, 3, figsize=(17, 4), constrained_layout=True)
axes[0].plot(train_df['update'], train_df['loss']); axes[0].set(title='Training loss',xlabel='Update')
long = val_df.loc[val_df['iteration']==400]
axes[1].plot(long['update'],long['micro_instance_f1'],marker='o'); axes[1].set(title='Validation instance F1@400',xlabel='Update')
for name in model.template_names:
    axes[2].plot(train_df['update'],train_df[name],label=name.removesuffix('.png'))
axes[2].set(title='Template strengths',xlabel='Update'); axes[2].legend(fontsize=7)
for ax in axes: ax.grid(alpha=.25)
plt.show()

## 7. Visualize learned refinement on a selected held-out slice

In [ ]:
from synthetic_fluor_stains_three_classes_weighted import SyntheticFluorStains
from unet_inference import load_unet, predict_volume
from mm_fixed_compatible import target_from_stain_labels
from template_energy import anchored_rollout
from instance_metrics import instance_stats
import numpy as np

VIS_DOMAIN = 'b'  # a, b, c
VIS_SEED = 400000
VIS_VOLUME = 0
VIS_SLICE = 8
VIS_ITERATIONS = 400
VIS_CHECKPOINT = 'best.pt'  # or latest.pt

state = torch.load(Path(cfg.output_dir) / VIS_CHECKPOINT,
                   map_location='cpu', weights_only=False)
prior_cfg = Config(**state['model_config'])
saved_cfg = state['training_config']
energy = TemplateEnergy(prior_cfg,
                        template_dir=PROJECT/'templates',
                        orientations=saved_cfg['orientations'],
                        strength_cap=saved_cfg['strength_cap'],
                        initial_strength=saved_cfg['initial_strength']).to(cfg.device)
assert tuple(state['template_names']) == energy.template_names, 'Template list/order differs from checkpoint'
energy.load_state_dict(state['model_state_dict'])
energy.eval().requires_grad_(False)

ds = SyntheticFluorStains(samples_per_epoch=VIS_VOLUME+1, seed=VIS_SEED,
                           N=cfg.N,z_slices=cfg.z_slices,z_ratio=cfg.z_ratio,
                           n_seeds=cfg.n_seeds,class_probs=cfg.class_probs,domain=VIS_DOMAIN)
images, labels = ds[VIS_VOLUME]
unet,_ = load_unet(cfg.checkpoint,cfg.device)
probs = predict_volume(unet, torch.as_tensor(images).float(), cfg.device, cfg.inference_batch_size)
del unet
raw = probs[VIS_SLICE:VIS_SLICE+1].to(cfg.device).detach()
target = target_from_stain_labels(torch.as_tensor(labels))[VIS_SLICE].numpy()
refined = anchored_rollout(energy, raw, raw, VIS_ITERATIONS,
                            cfg.anchor_mu, cfg.step_size, differentiable=False)
raw_np, refined_np = raw[0].cpu().numpy(), refined[0].cpu().numpy()

fig, axes = plt.subplots(4, 3, figsize=(12, 14), constrained_layout=True)
for col,(name,p) in enumerate([('GT',None),('Frozen U-Net',raw_np),('Template energy',refined_np)]):
    hard = target if p is None else p.argmax(0)
    axes[0,col].imshow(hard, vmin=0,vmax=2,cmap='viridis',interpolation='nearest')
    axes[0,col].set_title(name)
    if p is not None:
        st = instance_stats(hard,target)
        axes[0,col].set_xlabel(f'F1={st["instance_f1"]:.3f}, merges={st["merge_count"]}, splits={st["split_count"]}')
    for k in range(3):
        channel = (target == k).astype(float) if p is None else p[k]
        axes[k+1,col].imshow(channel, vmin=0,vmax=1,cmap='viridis',interpolation='nearest')
    for ax in axes[:,col]: ax.set_xticks([]);ax.set_yticks([])
for i, title in enumerate(['Hard labels','Background','Nuclear','Nonnuclear']):
    axes[i,0].set_ylabel(title)
plt.show()

## Notes / safeguards

- **Indigo:** coefficients are zero at unspecified pixels in the unrotated canonical template. Rotated filters use bilinear interpolation and can have small coefficients at pixels adjacent to the specified support; there is **no global mean subtraction**.
- **Color coding:** black contributes `(+1,-1)` and white contributes `(-1,+1)` on nuclear/nonnuclear channels. Background is not explicitly encoded by the painted colors. The existing MM potential and the U-Net anchor still include background.
- **Template order:** sorted filenames determine strength labels and are stored in checkpoints. Keep the same template folder for restoring checkpoints.
- **Finite training vs long validation:** 4 differentiable steps are used after randomized detached warm-ups; evaluate F1, merges, and splits after 400 steps as before.
- **No trainable kernel pixels:** all templates and rotations are fixed buffers. The only trainable parameters are eight bounded strengths. Existing MM geometry is unchanged.
